# Hands-on Build: Complete AI Agent

Now we will build a complete AI agent step-by-step using LangGraph.

In this project, our agent will:
- take user input
- decide what action to take
- use either an LLM or a tool
- store memory
- return a final response

This is how real AI workflow systems are designed.

## Planning the Agent

Before writing code,
we first plan the workflow.

Our agent needs to:

1. Accept user input
2. Decide whether to use:
   - LLM
   - or Tool
3. Generate response
4. Save conversation into memory
5. Return final output

## Workflow Design

The workflow looks like this:
```

Input
   ↓
Decide
   ↓
[LLM or Tool]
   ↓
Memory Update
   ↓
Output
```

This is a decision-based workflow,
not just a simple linear chain.

## Building Components
Define state, nodes, and edges.

In [ ]:
# Define state
class AgentState:
    user_input: str
    response: str
    memory: list
    need_tool: bool
    confidence: float

In [11]:
# Nodes
def input_node(state):
    # Simulate user input
    # state["user_input"] = input("Enter your query: ")
    return state

def decide_node(state):
    # Decide if need tool
    keywords = ["faq", "help", "support", "tool"]
    state["need_tool"] = any(word in state["user_input"].lower() for word in keywords)
    state["confidence"] = 0.8 if state["need_tool"] else 0.6
    return state

def llm_node(state):
    # Simulate LLM response
    context = "\n".join([f"Q: {m['input']} A: {m['response']}" for m in state['memory'][-2:]])
    state["response"] = f"LLM Answer (context: {context[:50]}...): {state['user_input']}"
    return state

def tool_node(state):
    # Simulate tool call
    faq_responses = {
        "faq": "FAQs: Visit our website for common questions.",
        "help": "Help: Contact support@company.com",
        "support": "Support: We're here to help!",
        "tool": "Tool: Calculator activated."
    }
    for key, response in faq_responses.items():
        if key in state["user_input"].lower():
            state["response"] = response
            break
    else:
        state["response"] = "Tool: No specific help found."
    return state

def memory_node(state):
    state["memory"].append({"input": state["user_input"], "response": state["response"]})
    return state

```
connect all node together
input->decide-> tool or llm ->memory->end
```


# Conditional Routing
This routing function controls the workflow path.

- If tool is needed → go to tool node
- Otherwise → go to LLM node

In [12]:
# Router function
def router(state):

    # Decide next node
    return "tool" if state["need_tool"] else "llm"

## Testing Components

Before running the full graph,
we test individual nodes separately.

This helps us:
- find bugs
- verify outputs
- debug faster

In [13]:

# Test decide_node
test_state = {"user_input": "I need help", "need_tool": False, "confidence": 0}
result = decide_node(test_state)
print("Decide result:", result)

# Test tool_node
test_state2 = {"user_input": "faq", "response": ""}
result2 = tool_node(test_state2)
print("Tool result:", result2)

Decide result: {'user_input': 'I need help', 'need_tool': True, 'confidence': 0.8}
Tool result: {'user_input': 'faq', 'response': 'FAQs: Visit our website for common questions.'}


## Common Issues

1. State not updating
   → Check return statement

2. Nodes not connecting
   → Verify node names

3. Conditional routing failing
   → Check routing function output

## Full Agent Assembly
Put it all together.

In [15]:
# pip install -U langgraph

from langgraph.graph import StateGraph, END

# Graph
graph = StateGraph(dict)

# Add nodes
graph.add_node("input", input_node)
graph.add_node("decide", decide_node)
graph.add_node("llm", llm_node)
graph.add_node("tool", tool_node)
graph.add_node("memory", memory_node)

# Entry
graph.set_entry_point("input")

# Edges
graph.add_edge("input", "decide")

# Router function
def router(state):
    return "tool" if state["need_tool"] else "llm"

# Conditional routing
graph.add_conditional_edges(
    "decide",
    router,
    {
        "tool": "tool",
        "llm": "llm"
    }
)

# Next steps
graph.add_edge("llm", "memory")
graph.add_edge("tool", "memory")

graph.add_edge("memory", END)

# Compile
app = graph.compile()

# Run
initial_state = {
    "user_input": "What is AI?",
    "response": "",
    "memory": [],
    "need_tool": False,
    "confidence": 0.0
}

# Invoke graph
result = app.invoke(initial_state)

print(result)

{'user_input': 'What is AI?', 'response': 'LLM Answer (context: ...): What is AI?', 'memory': [{'input': 'What is AI?', 'response': 'LLM Answer (context: ...): What is AI?'}], 'need_tool': False, 'confidence': 0.6}


```
 input -> decision-> tool/llm-> memory update-> end
```

## Unit Tests for Nodes
Write simple tests.

In [16]:
# Simple tests
def test_decide_node():
    state = {"user_input": "help me", "need_tool": False}
    result = decide_node(state)
    assert result["need_tool"] == True
    print("Decide test passed")

def test_memory_node():
    state = {"user_input": "test", "response": "answer", "memory": []}
    result = memory_node(state)
    assert len(result["memory"]) == 1
    print("Memory test passed")

test_decide_node()
test_memory_node()

Decide test passed
Memory test passed


## Summary

In this project, we built a complete AI workflow system using LangGraph.

We learned:
- state management
- nodes
- conditional routing
- memory handling
- graph assembly
- testing

This is the foundation of real AI agents and automation systems.